# Self-Supervised Learning

Companion notebook for the [Self-Supervised Learning lesson](https://ml-viz.vercel.app/courses/computer-vision/05-self-supervised-learning).

We implement the **InfoNCE contrastive loss** with in-batch negatives, demonstrate **representation
collapse** (and why negatives prevent it), and build a tiny **masked-reconstruction** pretext task.
Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

def normalize(Z):
    return Z / np.linalg.norm(Z, axis=1, keepdims=True)

## 1 — InfoNCE contrastive loss

For a batch, view A and view B are two augmentations of the same images (so row i of A matches row i
of B). The loss pulls each A toward its matching B and pushes it from all other B's — a softmax over
similarities with the diagonal as the target.

In [ ]:
def info_nce(Za, Zb, tau=0.1):
    Za, Zb = normalize(Za), normalize(Zb)
    logits = (Za @ Zb.T) / tau          # (B,B) similarity matrix
    logits -= logits.max(1, keepdims=True)
    p = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
    B = len(Za)
    return -np.mean(np.log(p[np.arange(B), np.arange(B)] + 1e-9))

B, d = 8, 16
base = rng.normal(size=(B, d))
Za = base + 0.05 * rng.normal(size=(B, d))     # two augmented views of the SAME images
Zb = base + 0.05 * rng.normal(size=(B, d))
Zrand = rng.normal(size=(B, d))                 # unrelated
print(f'loss, matched views:   {info_nce(Za, Zb):.3f}  (low: positives align)')
print(f'loss, mismatched views: {info_nce(Zrand, Zb):.3f}  (high)')

## 2 — Representation collapse

A 'shortcut' encoder that maps everything to the same vector makes positives perfectly aligned — but
the *negatives* in InfoNCE punish it, because then every pair is equally similar and the softmax
can't separate the true positive. Negatives are what prevent collapse.

In [ ]:
collapsed = np.ones((B, d))                      # encoder output: identical for all inputs
print(f'InfoNCE on collapsed reps: {info_nce(collapsed, collapsed):.3f}')
print(f'InfoNCE on good reps:      {info_nce(Za, Zb):.3f}')
print(f'log(B) = {np.log(B):.3f}  <- collapsed loss hits this floor: every pair looks identical')
print('\nThe negatives make collapse a HIGH-loss solution, so the model avoids it.')

## 3 — A masked-reconstruction pretext task

Masked modeling (MAE/BERT) hides part of the input and predicts it. We mask patches of a signal and
measure reconstruction error — a model that learns structure reconstructs better than the mean
baseline.

In [ ]:
def mask_patches(x, mask_ratio, seed):
    r = np.random.default_rng(seed)
    mask = r.random(len(x)) < mask_ratio
    visible = x.copy(); visible[mask] = 0.0
    return visible, mask

x = np.sin(np.linspace(0, 6*np.pi, 60))         # a structured signal
visible, mask = mask_patches(x, mask_ratio=0.5, seed=1)
# a 'model' that knows the structure interpolates the masked points; baseline predicts the mean
recon = visible.copy(); recon[mask] = np.interp(np.where(mask)[0], np.where(~mask)[0], x[~mask])
err_model = np.mean((recon[mask] - x[mask])**2)
err_base  = np.mean((x.mean() - x[mask])**2)
print(f'reconstruction MSE  (structure-aware): {err_model:.3f}')
print(f'reconstruction MSE  (mean baseline):   {err_base:.3f}')
print('Lower error means the representation captured the signal structure -> useful features.')

## ✏️ Your turn

**Exercise.** Implement `cosine_sim_matrix(Za, Zb)` (pairwise cosine similarities between two batches
of L2-normalized rows) and `contrastive_accuracy(Za, Zb)` — the fraction of rows whose *most similar*
partner in Zb is its true match (the diagonal). This is the standard retrieval-style SSL probe.

In [ ]:
def cosine_sim_matrix(Za, Zb):
    # TODO(you): normalize rows, return the (B,B) matrix of cosine similarities Za_i . Zb_j
    return ...

def contrastive_accuracy(Za, Zb):
    # TODO(you): fraction of rows i whose argmax over Zb is i (the true positive)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
S = cosine_sim_matrix(Za, Zb)
assert S.shape == (B, B)
assert np.allclose(np.diag(cosine_sim_matrix(Za, Za)), 1.0)     # self-similarity is 1
assert contrastive_accuracy(Za, Zb) == 1.0                       # matched views retrieve each other
assert contrastive_accuracy(Zrand, Zb) < 1.0                     # random views don't
print('\u2713 similarity matrix and contrastive accuracy are correct')

<details>
<summary>Solution</summary>

```python
def cosine_sim_matrix(Za, Zb):
    return normalize(Za) @ normalize(Zb).T

def contrastive_accuracy(Za, Zb):
    S = cosine_sim_matrix(Za, Zb)
    return np.mean(S.argmax(axis=1) == np.arange(len(Za)))
```

Pulling matched views together (high diagonal similarity) while pushing apart mismatches is the
whole game; the negatives are what stop the trivial collapse solution from winning.

</details>